In [ ]:

# -*- coding: utf-8 -*-
"""
PI Web API extraction with aligned timestamps.

Main features
-------------
- Lock "now" once and align to the interval boundary
- Use the same absolute start/end window for all tags
- Keep output format: UTC index + TS_LOCAL + TS_UTC
- Cache tag index as <data_server_name>_index.xlsx
- Extract tags in parallel (max 5 workers; notify admin before using more)
- If a tag fails:
    * wait 10 seconds
    * retry the same tag
    * reduce parallel workers
- After 5 attempts, mark tag as unable to extract
- Save output as .parquet (fast, low memory)
- Separate utility to convert parquet -> xlsx
- Write a live .log file with tag path and request URL while running
- Prevent OS sleep (Windows + macOS) during extraction
- 20 ms delay between paging requests to avoid server overload

Important fixes
---------------
- Preserves non-numeric PI tags (Digital / String / state-like values)
- Keeps ALL requested tags in the final table, even if they return no usable rows
- Builds a master aligned timestamp index so empty tags still appear as columns

author: franktoffel (github.com/franktoffel)
license: MIT
"""

from __future__ import annotations

import re
import sys
import time
import json
import math
import getpass
import subprocess
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import requests
import urllib3
from requests.adapters import HTTPAdapter
from requests_negotiate_sspi import HttpNegotiateAuth

try:
    from IPython.display import clear_output
except Exception:
    clear_output = None


# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
_MAX_SAFE_WORKERS = 5

NUMERIC_POINT_TYPES = {
    "Float16", "Float32", "Float64",
    "Int8", "Int16", "Int32", "Int64",
    "UInt8", "UInt16", "UInt32", "UInt64",
    "Single", "Double",
}


# ---------------------------------------------------------------------------
# Keep-awake: prevent OS sleep during extraction
# ---------------------------------------------------------------------------
class _KeepAwake:
    """
    Prevent the OS from sleeping during extraction.
    - Windows : uses ctypes SetThreadExecutionState
    - macOS   : spawns 'caffeinate' subprocess
    - Linux   : no-op (handled by systemd/settings)
    """

    def __init__(self):
        self._proc = None

    def start(self):
        if sys.platform == "win32":
            import ctypes
            ctypes.windll.kernel32.SetThreadExecutionState(0x80000003)
            print("[keep-awake] Windows: sleep prevented.")
        elif sys.platform == "darwin":
            self._proc = subprocess.Popen(
                ["caffeinate", "-i"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            print("[keep-awake] macOS: caffeinate started.")
        else:
            print("[keep-awake] Linux: no-op, manage via system settings.")

    def stop(self):
        if sys.platform == "win32":
            import ctypes
            ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)
            print("[keep-awake] Windows: sleep restored.")
        elif sys.platform == "darwin":
            if self._proc is not None:
                self._proc.terminate()
                self._proc = None
            print("[keep-awake] macOS: caffeinate stopped.")


# ---------------------------------------------------------------------------
# Auth and sessions
# ---------------------------------------------------------------------------
_basic_credentials: dict[str, tuple[str, str]] = {}
_session_cache: dict[str, requests.Session] = {}
_thread_local = threading.local()


def _prompt_credentials(hostname: str) -> tuple[str, str]:
    if hostname not in _basic_credentials:
        print(f"\n[AUTH] Server '{hostname}' requires explicit login (email/password).")
        user = input("  Email / Username: ").strip()
        pwd = getpass.getpass("  Password: ")
        _basic_credentials[hostname] = (user, pwd)
    return _basic_credentials[hostname]


def _make_session(verify: bool = False, server_url: str = "") -> requests.Session:
    from urllib.parse import urlparse

    hostname = urlparse(server_url).hostname or ""

    if hostname in _session_cache:
        return _session_cache[hostname]

    s = requests.Session()
    s.verify = verify

    if not verify:
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    s.auth = HttpNegotiateAuth()

    if server_url:
        probe = s.get(
            server_url.rstrip("/") + "/system/versions",
            verify=verify,
            timeout=15,
            allow_redirects=True,
        )
        if probe.status_code == 401:
            user, pwd = _prompt_credentials(hostname)
            s.auth = (user, pwd)

    _session_cache[hostname] = s
    return s


def _make_thread_session(
    verify: bool = False,
    server_url: str = "",
    pool_maxsize: int = 20,
) -> requests.Session:
    from urllib.parse import urlparse

    hostname = urlparse(server_url).hostname or ""

    if not hasattr(_thread_local, "sessions"):
        _thread_local.sessions = {}

    key = (hostname, bool(verify), int(pool_maxsize))
    if key in _thread_local.sessions:
        return _thread_local.sessions[key]

    s = requests.Session()
    s.verify = verify

    if not verify:
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    adapter = HTTPAdapter(pool_connections=pool_maxsize, pool_maxsize=pool_maxsize)
    s.mount("http://", adapter)
    s.mount("https://", adapter)

    if hostname in _basic_credentials:
        s.auth = _basic_credentials[hostname]
    else:
        s.auth = HttpNegotiateAuth()
        if server_url:
            probe = s.get(
                server_url.rstrip("/") + "/system/versions",
                verify=verify,
                timeout=15,
                allow_redirects=True,
            )
            if probe.status_code == 401:
                user, pwd = _prompt_credentials(hostname)
                s.auth = (user, pwd)

    _thread_local.sessions[key] = s
    return s


# ---------------------------------------------------------------------------
# Live extraction log
# ---------------------------------------------------------------------------
_log_lock = threading.Lock()
_log_fp = None
_log_path = None


def _open_live_log(log_file: str):
    global _log_fp, _log_path
    _log_path = str(Path(log_file))
    Path(_log_path).parent.mkdir(parents=True, exist_ok=True)
    _log_fp = open(_log_path, mode="w", encoding="utf-8", buffering=1)
    _log_fp.write("timestamp_utc\tevent\tattempt\tname\tlabel\tpath\turl\tmessage\n")
    _log_fp.flush()


def _close_live_log():
    global _log_fp
    if _log_fp is not None:
        try:
            _log_fp.flush()
            _log_fp.close()
        except Exception:
            pass
        _log_fp = None


def _write_live_log(
    event: str,
    *,
    attempt: int | None = None,
    name: str = "",
    label: str = "",
    path: str = "",
    url: str = "",
    message: str = "",
):
    global _log_fp
    if _log_fp is None:
        return

    ts = pd.Timestamp.now("UTC").strftime("%Y-%m-%d %H:%M:%S")

    def _clean(x):
        if x is None:
            return ""
        return str(x).replace("\t", " ").replace("\n", " ").replace("\r", " ")

    line = "\t".join([
        _clean(ts), _clean(event), _clean(attempt), _clean(name),
        _clean(label), _clean(path), _clean(url), _clean(message),
    ]) + "\n"

    with _log_lock:
        _log_fp.write(line)
        _log_fp.flush()


# ---------------------------------------------------------------------------
# Small helpers
# ---------------------------------------------------------------------------
def _fmt_duration(seconds: float | None) -> str:
    if seconds is None:
        return "--:--:--"
    seconds = max(0, int(round(seconds)))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def _point_type_is_numeric(point_type: str | None) -> bool:
    if point_type is None:
        return False
    return str(point_type).strip() in NUMERIC_POINT_TYPES


def _serialize_non_numeric_value(v):
    """
    Preserve digital/string-like PI values safely.
    """
    if v is None:
        return None

    # PI digital values often come as objects
    if isinstance(v, dict):
        # Prefer human-readable state name when present
        if "Name" in v and v["Name"] is not None:
            return str(v["Name"])
        if "Value" in v and v["Value"] is not None:
            return str(v["Value"])
        try:
            return json.dumps(v, ensure_ascii=False, sort_keys=True)
        except Exception:
            return str(v)

    if isinstance(v, (list, tuple)):
        try:
            return json.dumps(v, ensure_ascii=False)
        except Exception:
            return str(v)

    return v


def _extract_pi_value(raw_value, point_type: str | None):
    """
    Normalize PI value depending on tag type.

    Numeric tags:
        - try to extract numeric component and convert to float
    Non-numeric tags:
        - preserve human-readable text / digital state / string
    """
    if _point_type_is_numeric(point_type):
        candidate = raw_value

        # Numeric values may occasionally come inside a dict
        if isinstance(raw_value, dict):
            if "Value" in raw_value:
                candidate = raw_value.get("Value")
            elif "Name" in raw_value:
                candidate = raw_value.get("Name")

        val = pd.to_numeric(pd.Series([candidate]), errors="coerce").iloc[0]
        return val

    # Non-numeric: preserve as text / object
    return _serialize_non_numeric_value(raw_value)


def _empty_value_array(length: int, point_type: str | None):
    if _point_type_is_numeric(point_type):
        return np.full(length, np.nan, dtype=float)
    return np.array([None] * length, dtype=object)


# ---------------------------------------------------------------------------
# 1) Tag index
# ---------------------------------------------------------------------------
def _index_path(data_server_name: str) -> Path:
    safe_name = re.sub(r"[^\w\-]", "_", data_server_name)
    return Path(f"{safe_name}_index.xlsx")


def build_index(
    server: str,
    data_server_name: str,
    *,
    max_results: int = 200_000,
    verify: bool = False,
    verbose: bool = False,
) -> pd.DataFrame:
    base = server.rstrip("/")
    session = _make_session(verify=verify, server_url=base)

    r = session.get(f"{base}/dataservers", params={"name": data_server_name}, timeout=60)
    if verbose:
        print("GET", r.request.url)
    r.raise_for_status()

    ds = r.json()
    if "WebId" in ds:
        dataserver_webid = ds["WebId"]
    elif isinstance(ds, dict) and "Items" in ds and ds["Items"]:
        dataserver_webid = ds["Items"][0]["WebId"]
    else:
        raise RuntimeError(f"Data Server '{data_server_name}' not found at {server}.")

    selected_fields = (
        "Items.WebId;Items.Name;Items.Path;Items.Descriptor;"
        "Items.PointClass;Items.PointType;Items.EngineeringUnits;Items.Step;Items.Future"
    )

    page_size = 1000
    max_results = int(max_results)
    url = f"{base}/dataservers/{dataserver_webid}/points"
    rows = []
    start_index = 0

    print(f"[build_index] Enumerating all points on '{data_server_name}' ...")

    while len(rows) < max_results:
        page_count = min(page_size, max_results - len(rows))
        params = {
            "startIndex": start_index,
            "maxCount": page_count,
            "selectedFields": selected_fields,
        }

        resp = session.get(url, params=params, timeout=120)
        if verbose:
            print(f"  GET {resp.request.url} -> fetched so far: {len(rows)}")
        resp.raise_for_status()

        items = resp.json().get("Items", [])
        if not items:
            break

        rows.extend(items)
        start_index += len(items)

        if len(items) < page_count:
            break

        time.sleep(0.02)

    if not rows:
        print("[build_index] WARNING: No points found.")
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    if "WebId" in df.columns:
        df = df.drop_duplicates(subset="WebId", keep="first")
    df = df.reset_index(drop=True)

    idx_file = _index_path(data_server_name)
    df.to_excel(idx_file, index=False)
    print(f"[build_index] {len(df)} points saved -> {idx_file}")
    return df


def load_index(
    data_server_name: str,
    server: str = "",
    *,
    reindex: bool = False,
    max_results: int = 200_000,
    verify: bool = False,
    verbose: bool = False,
) -> pd.DataFrame:
    idx_file = _index_path(data_server_name)

    if not reindex and idx_file.exists():
        print(f"[load_index] Loading cached index from '{idx_file}' ...")
        df = pd.read_excel(idx_file, dtype=str, engine="openpyxl")
        print(f"[load_index] {len(df)} points loaded.")
        return df

    if not server:
        raise ValueError("'server' URL is required when no index file exists or reindex=True.")

    return build_index(server, data_server_name, max_results=max_results, verify=verify, verbose=verbose)


def filter_index(
    df: pd.DataFrame,
    *,
    name_filter: str | None = None,
    description_filter: str | None = None,
) -> pd.DataFrame:

    def _pi_glob_to_regex(pattern: str) -> str:
        regex = ""
        for ch in pattern:
            if ch == "*":
                regex += ".*"
            elif ch == "?":
                regex += "."
            elif ch in r"\.^$+{}[]|()":
                regex += "\\" + ch
            else:
                regex += ch
        return f"(?si)^{regex}$"

    def _apply(series: pd.Series, pattern: str, auto_substring: bool = False) -> pd.Series:
        p = pattern.strip()
        if p == "*":
            return pd.Series(True, index=series.index)
        if auto_substring and "*" not in p and "?" not in p:
            p = f"*{p}*"
        rx = _pi_glob_to_regex(p)
        return series.fillna("").str.match(rx)

    result = df.copy()

    if name_filter is not None:
        mask = _apply(result["Name"], name_filter, auto_substring=False)
        kept, dropped = int(mask.sum()), int((~mask).sum())
        if dropped:
            print(f"[filter_index] name_filter '{name_filter}': {kept} kept / {dropped} dropped.")
        else:
            print(f"[filter_index] name_filter '{name_filter}': keep all {kept} (no filter applied).")
        result = result[mask].copy()

    if description_filter is not None:
        mask = _apply(result["Descriptor"], description_filter, auto_substring=True)
        kept, dropped = int(mask.sum()), int((~mask).sum())
        if dropped:
            print(f"[filter_index] description_filter '{description_filter}': {kept} kept / {dropped} dropped.")
        else:
            print(f"[filter_index] description_filter '{description_filter}': keep all {kept} (no filter applied).")
        result = result[mask].copy()

    return result.reset_index(drop=True)


# ---------------------------------------------------------------------------
# 2) Single tag extraction
# ---------------------------------------------------------------------------
def get_pi_data(
    server,
    tag_path,
    start="*-1y",
    end="*",
    interval="1d",
    verify=False,
    verbose=False,
    good_only=True,
    session: requests.Session | None = None,
    tag_name: str = "",
    tag_label: str = "",
    attempt: int | None = None,
    paging_delay_ms: int = 20,
    point_type: str | None = None,
):
    """
    Fetch one tag's interpolated data.

    Returns DataFrame indexed by UTC_Index with columns:
        TS_LOCAL, TS_UTC, Value, Good, Questionable
    """
    base = server.rstrip("/")
    s = session or _make_session(verify=verify, server_url=base)
    s.verify = verify

    if not verify:
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    point_req = requests.Request("GET", f"{base}/points", params={"path": tag_path}).prepare()
    point_lookup_url = point_req.url

    try:
        r = s.get(f"{base}/points", params={"path": tag_path}, timeout=60)
        if verbose:
            print("GET", r.request.url)
        r.raise_for_status()

        body = r.json()
        webid = body["WebId"]

        params = {
            "startTime": start,
            "endTime": end,
            "interval": interval,
            "selectedFields": "Items.Timestamp;Items.Value;Items.Good;Items.Questionable;Links.Next",
        }

        if good_only:
            params["filterExpression"] = "BadVal('.')=0"
            params["includeFilteredValues"] = "false"

        stream_req = requests.Request(
            "GET", f"{base}/streams/{webid}/interpolated", params=params
        ).prepare()
        stream_url = stream_req.url

        _write_live_log(
            "RUNNING",
            attempt=attempt,
            name=tag_name,
            label=tag_label,
            path=tag_path,
            url=stream_url,
            message="starting extraction",
        )

        # Clean paging loop
        all_items = []
        next_url = f"{base}/streams/{webid}/interpolated"
        next_params = dict(params)

        while next_url:
            time.sleep(paging_delay_ms / 1000.0)
            r = s.get(next_url, params=next_params, timeout=300)
            if verbose:
                print("GET", r.request.url)
            r.raise_for_status()

            body = r.json()
            items = body.get("Items", [])
            all_items.extend(items)

            next_url = body.get("Links", {}).get("Next")
            next_params = None  # 'Next' already contains all paging/query params

        df = pd.DataFrame(all_items)
        empty_df = pd.DataFrame(columns=["TS_LOCAL", "TS_UTC", "Value", "Good", "Questionable"])

        if df.empty:
            _write_live_log(
                "SUCCESS",
                attempt=attempt,
                name=tag_name,
                label=tag_label,
                path=tag_path,
                url=stream_url,
                message="no rows returned",
            )
            return empty_df

        if "Good" not in df.columns:
            df["Good"] = True
        if "Questionable" not in df.columns:
            df["Questionable"] = False

        # Preserve or convert value depending on point type
        if "Value" in df.columns:
            df["Value"] = df["Value"].apply(lambda v: _extract_pi_value(v, point_type))
        else:
            df["Value"] = np.nan if _point_type_is_numeric(point_type) else None

        # Apply filtering
        if good_only:
            df = df[df["Good"].fillna(False)].copy()
            df = df[~df["Questionable"].fillna(False)].copy()

            # Only require Value.notna() for numeric tags
            if _point_type_is_numeric(point_type):
                df = df[df["Value"].notna()].copy()

        if df.empty:
            _write_live_log(
                "SUCCESS",
                attempt=attempt,
                name=tag_name,
                label=tag_label,
                path=tag_path,
                url=stream_url,
                message="all rows filtered out",
            )
            return empty_df

        ts = pd.to_datetime(df["Timestamp"], errors="coerce", utc=True)
        good_ts = ts.notna()
        df = df.loc[good_ts].copy()
        ts = ts.loc[good_ts]

        if df.empty:
            _write_live_log(
                "SUCCESS",
                attempt=attempt,
                name=tag_name,
                label=tag_label,
                path=tag_path,
                url=stream_url,
                message="no valid timestamps",
            )
            return empty_df

        ts_utc_floor = ts.dt.floor("s")
        ts_local_floor = ts_utc_floor.dt.tz_localize(None)

        df["TS_UTC"] = ts_utc_floor.dt.strftime("%Y-%m-%d %H:%M:%S")
        df["TS_LOCAL"] = ts_local_floor.dt.strftime("%Y-%m-%d %H:%M:%S")
        df = df.set_index(ts_utc_floor).sort_index()
        df.index.name = "UTC_Index"

        # In case of duplicate timestamps, keep last
        df = df[~df.index.duplicated(keep="last")]

        _write_live_log(
            "SUCCESS",
            attempt=attempt,
            name=tag_name,
            label=tag_label,
            path=tag_path,
            url=stream_url,
            message=f"rows={len(df)}",
        )

        cols = ["TS_LOCAL", "TS_UTC", "Value", "Good", "Questionable"]
        return df[[c for c in cols if c in df.columns]]

    except Exception as ex:
        _write_live_log(
            "ERROR",
            attempt=attempt,
            name=tag_name,
            label=tag_label,
            path=tag_path,
            url=point_lookup_url,
            message=str(ex),
        )
        raise


# ---------------------------------------------------------------------------
# 3) Interval helpers
# ---------------------------------------------------------------------------
_INTERVAL_RE = re.compile(r"^\s*(\d+)\s*([smhdSMHD])\s*$")
_REL_RE = re.compile(r"^\s*\*\s*-\s*(\d+)\s*([smhdSMHD])\s*$")


def _parse_interval(interval: str) -> tuple[pd.Timedelta, str]:
    m = _INTERVAL_RE.match(interval)
    if not m:
        raise ValueError(f"Unsupported interval: {interval!r}")
    n, unit = int(m.group(1)), m.group(2).lower()
    freq_map = {
        "s": (pd.to_timedelta(n, unit="s"), f"{n}s"),
        "m": (pd.to_timedelta(n, unit="m"), f"{n}min"),
        "h": (pd.to_timedelta(n, unit="h"), f"{n}h"),
        "d": (pd.to_timedelta(n, unit="D"), f"{n * 24}h"),
    }
    if unit not in freq_map:
        raise ValueError(f"Unsupported unit: {unit!r}")
    return freq_map[unit]


def _to_pi_interval(interval: str) -> str:
    m = _INTERVAL_RE.match(interval)
    if not m:
        raise ValueError(f"Unsupported interval: {interval!r}")
    n, unit = int(m.group(1)), m.group(2).lower()
    return {"s": f"{n}s", "m": f"{n}m", "h": f"{n}h", "d": f"{n * 24}h"}[unit]


def _anchor_end_time(interval: str, sync_time: str | None = None) -> pd.Timestamp:
    _td, freq = _parse_interval(interval)
    now_utc = pd.Timestamp.now(tz="UTC").floor("min")
    if sync_time is None:
        return now_utc.floor(freq)
    hh, mm, ss = map(int, sync_time.strip().split(":"))
    today_sync = now_utc.floor("D").replace(
        hour=hh, minute=mm, second=ss, microsecond=0, nanosecond=0
    )
    if now_utc >= today_sync:
        return today_sync
    return today_sync - pd.Timedelta(days=1)


def _absolute_window(start: str, end_anchor_utc: pd.Timestamp) -> tuple[str, str]:
    end_str = end_anchor_utc.strftime("%Y-%m-%dT%H:%M:%SZ")
    if start is None or start.strip() == "*":
        return end_str, end_str
    m = _REL_RE.match(start)
    if m:
        n, unit = int(m.group(1)), m.group(2).lower()
        td_unit = "D" if unit == "d" else unit
        start_utc = end_anchor_utc - pd.to_timedelta(n, unit=td_unit)
        return start_utc.strftime("%Y-%m-%dT%H:%M:%SZ"), end_str
    return start, end_str


def _build_master_index(start_abs: str, end_abs: str, interval: str) -> pd.DatetimeIndex:
    _td, freq = _parse_interval(interval)
    start_ts = pd.Timestamp(start_abs, tz="UTC")
    end_ts = pd.Timestamp(end_abs, tz="UTC")
    return pd.date_range(
        start=start_ts,
        end=end_ts,
        freq=freq,
        tz="UTC",
        name="UTC_Index",
    )


# ---------------------------------------------------------------------------
# 4) Labels and per-tag wrapper
# ---------------------------------------------------------------------------
def build_label(row: pd.Series) -> str:
    name = str(row.get("Name", "")).strip()
    desc = str(row.get("Descriptor") or "").strip()
    eu = str(row.get("EngineeringUnits") or "").strip()
    return f"'{name}' ('{desc}') ['{eu}']"


def fetch_one_point(
    server,
    tag_path,
    label,
    start_abs,
    end_abs,
    interval,
    verify=False,
    verbose=False,
    session=None,
    tag_name="",
    attempt=None,
    paging_delay_ms: int = 20,
    point_type: str | None = None,
    master_index: pd.DatetimeIndex | None = None,
) -> pd.DataFrame:
    """
    Always returns a DataFrame indexed by the full master_index,
    with columns TS_LOCAL, TS_UTC, and the tag label.

    This guarantees that even empty tags still appear as columns.
    """
    pi_interval = _to_pi_interval(interval)
    if verbose:
        print(f"  [PI interval] {interval!r} -> {pi_interval!r}")

    idx = master_index
    if idx is None:
        idx = _build_master_index(start_abs, end_abs, interval)

    # Create full aligned output frame up front
    out = pd.DataFrame(index=idx)
    out.index.name = "UTC_Index"
    out["TS_UTC"] = out.index.strftime("%Y-%m-%d %H:%M:%S")
    out["TS_LOCAL"] = out.index.tz_localize(None).strftime("%Y-%m-%d %H:%M:%S")
    out[label] = _empty_value_array(len(out), point_type)

    df = get_pi_data(
        server,
        tag_path,
        start=start_abs,
        end=end_abs,
        interval=pi_interval,
        verify=verify,
        verbose=verbose,
        session=session,
        tag_name=tag_name,
        tag_label=label,
        attempt=attempt,
        paging_delay_ms=paging_delay_ms,
        point_type=point_type,
    )

    if df.empty:
        return out[["TS_LOCAL", "TS_UTC", label]]

    # Reindex to full master grid
    aligned = df.reindex(idx)

    if "Value" in aligned.columns:
        out[label] = aligned["Value"].values

    return out[["TS_LOCAL", "TS_UTC", label]]


# ---------------------------------------------------------------------------
# 5) Parquet save/load + Excel export
# ---------------------------------------------------------------------------
def save_to_parquet(
    wide: pd.DataFrame,
    errors: pd.DataFrame,
    output_parquet: str,
):
    """
    Save wide DataFrame and errors to parquet files.
    Data   -> <output_parquet>
    Errors -> <output_parquet stem>_errors.parquet
    """
    out = Path(output_parquet)
    out.parent.mkdir(parents=True, exist_ok=True)

    wide.to_parquet(out, engine="pyarrow")
    print(f"[save_to_parquet] Data saved -> {out}  ({wide.shape[0]} rows x {wide.shape[1]} cols)")

    if not errors.empty:
        err_path = out.with_name(out.stem + "_errors.parquet")
        errors.to_parquet(err_path, engine="pyarrow")
        print(f"[save_to_parquet] Errors saved -> {err_path}  ({len(errors)} rows)")
    else:
        print("[save_to_parquet] No errors to save.")


def parquet_to_excel(
    parquet_path: str,
    output_xlsx: str | None = None,
    errors_parquet_path: str | None = None,
    chunksize: int = 200_000,
):
    """
    Convert a parquet file produced by fetch_many_points to an Excel file.

    Because xlsx has a 1,048,576 row limit and large files will crash the
    browser/Excel, this function:
      - Writes in chunks if needed (one sheet per chunk)
      - Strips tz from the UTC index (Excel does not support tz-aware datetimes)
      - Writes errors on a separate sheet if errors parquet is found
    """
    src = Path(parquet_path)
    if output_xlsx is None:
        output_xlsx = str(src.with_suffix(".xlsx"))

    print(f"[parquet_to_excel] Reading {src} ...")
    wide = pd.read_parquet(src, engine="pyarrow")
    print(f"[parquet_to_excel] Shape: {wide.shape}")

    export = wide.reset_index()
    if "UTC_Index" in export.columns and hasattr(export["UTC_Index"], "dt"):
        if export["UTC_Index"].dt.tz is not None:
            export["UTC_Index"] = export["UTC_Index"].dt.tz_localize(None)

    n_rows = len(export)
    n_chunks = max(1, -(-n_rows // chunksize))

    print(f"[parquet_to_excel] Writing {n_rows} rows in {n_chunks} sheet(s) -> {output_xlsx} ...")

    with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
        for i in range(n_chunks):
            chunk = export.iloc[i * chunksize: (i + 1) * chunksize]
            sheet_name = "Data" if n_chunks == 1 else f"Data_{i + 1}"
            chunk.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"  Sheet '{sheet_name}': {len(chunk)} rows")

        err_src = errors_parquet_path
        if err_src is None:
            auto = src.with_name(src.stem + "_errors.parquet")
            if auto.exists():
                err_src = str(auto)

        if err_src and Path(err_src).exists():
            errors_df = pd.read_parquet(err_src, engine="pyarrow")
            errors_df.to_excel(writer, sheet_name="Errors", index=False)
            print(f"  Sheet 'Errors': {len(errors_df)} rows")

    print(f"[parquet_to_excel] Done -> {output_xlsx}")


# ---------------------------------------------------------------------------
# 6) Bulk fetch in parallel
# ---------------------------------------------------------------------------
def fetch_many_points(
    server,
    points_df: pd.DataFrame,
    start="*-30d",
    end="*",
    interval="1h",
    sync_time=None,
    verify=False,
    verbose=False,
    output_parquet: str | None = None,
    log_file: str | None = None,
    max_workers: int = 5,
    min_workers: int = 1,
    max_attempts: int = 5,
    retry_wait_seconds: int = 10,
    status_every: int = 5,
    paging_delay_ms: int = 20,
):
    """
    Fetch all tags in parallel into one aligned wide DataFrame.

    Guarantees:
    - every requested tag appears as a column
    - non-numeric tags are preserved
    - empty tags become all-NaN / empty columns
    """

    if max_workers > _MAX_SAFE_WORKERS:
        print(
            f"\n⚠️  WARNING: max_workers={max_workers} exceeds the recommended limit of "
            f"{_MAX_SAFE_WORKERS}.\n"
            f"   High parallelism can overload the PI server and affect other users.\n"
            f"   Please notify the PI administrator before proceeding.\n"
        )
        while True:
            answer = input(
                "   Have you notified the administrator? Type 'y' or 'yes' to continue, anything else to abort: "
            ).strip().lower()
            if answer in ("y", "yes"):
                print(f"   Proceeding with max_workers={max_workers}.\n")
                break
            raise RuntimeError(
                f"Extraction aborted. Please contact the PI administrator before "
                f"using more than {_MAX_SAFE_WORKERS} workers."
            )

        max_workers = _MAX_SAFE_WORKERS

    end_anchor_utc = _anchor_end_time(interval, sync_time)
    start_abs, end_abs = _absolute_window(start, end_anchor_utc)
    master_index = _build_master_index(start_abs, end_abs, interval)

    if log_file is None:
        if output_parquet:
            log_file = str(Path(output_parquet).with_suffix("")) + "_live.log"
        else:
            log_file = "pi_extraction_live.log"

    _open_live_log(log_file)
    _awake = _KeepAwake()
    _awake.start()

    try:
        _ = _make_session(verify=verify, server_url=server.rstrip("/"))

        def _show_status(text: str):
            try:
                if clear_output is not None:
                    clear_output(wait=True)
                else:
                    print("\033[2J\033[H", end="")
            except Exception:
                pass
            print(text)

        label_counts: dict[str, int] = {}

        def _uniq(label: str) -> str:
            cnt = label_counts.get(label, 0) + 1
            label_counts[label] = cnt
            return label if cnt == 1 else f"{label} #{cnt}"

        pending: list[dict] = []
        for _, row in points_df.iterrows():
            job = row.to_dict()
            job["_label"] = _uniq(build_label(row))
            job["_attempts"] = 0
            job["_ready_at"] = 0.0
            pending.append(job)

        total = len(pending)
        workers = max(int(max_workers), 1)
        min_workers = max(int(min_workers), 1)

        parts: list[pd.DataFrame] = []
        errors: list[dict] = []

        completed_ok = 0
        completed_fail = 0
        attempt_count = 0
        total_attempt_secs = 0.0
        ts_added = False

        t_start = time.perf_counter()
        last_status_attempts = 0

        if verbose:
            _show_status(
                f"[start]\n"
                f"tags={total}  workers={workers}\n"
                f"window: {start_abs} -> {end_abs}\n"
                f"paging_delay={paging_delay_ms}ms  retry_wait={retry_wait_seconds}s\n"
                f"log={log_file}"
            )

        def _worker(job: dict) -> dict:
            t0 = time.perf_counter()
            try:
                session = _make_thread_session(
                    verify=verify,
                    server_url=server.rstrip("/"),
                    pool_maxsize=max(20, max_workers * 2),
                )
                one = fetch_one_point(
                    server=server,
                    tag_path=job["Path"],
                    label=job["_label"],
                    start_abs=start_abs,
                    end_abs=end_abs,
                    interval=interval,
                    verify=verify,
                    verbose=False,
                    session=session,
                    tag_name=str(job.get("Name", "")),
                    attempt=int(job["_attempts"]) + 1,
                    paging_delay_ms=paging_delay_ms,
                    point_type=job.get("PointType"),
                    master_index=master_index,
                )
                return {"ok": True, "job": job, "df": one, "secs": time.perf_counter() - t0}
            except Exception as ex:
                return {"ok": False, "job": job, "error": str(ex), "secs": time.perf_counter() - t0}

        while pending:
            now = time.time()
            ready = [j for j in pending if j["_ready_at"] <= now]

            if not ready:
                next_ready = min(j["_ready_at"] for j in pending)
                sleep_for = max(0.0, next_ready - now)

                if verbose:
                    elapsed = time.perf_counter() - t_start
                    done_total = completed_ok + completed_fail
                    tags_per_min = done_total / elapsed * 60.0 if done_total > 0 and elapsed > 0 else 0.0
                    avg_tag_sec = elapsed / done_total if done_total > 0 else None
                    eta_sec = len(pending) * avg_tag_sec if avg_tag_sec else None
                    attempts_per_min = attempt_count / elapsed * 60.0 if elapsed > 0 else 0.0
                    avg_attempt_sec = total_attempt_secs / attempt_count if attempt_count > 0 else None

                    _show_status(
                        f"[waiting for retries]\n"
                        f"done={done_total}/{total}  ok={completed_ok}  failed={completed_fail}  pending={len(pending)}\n"
                        f"workers={workers}  attempts={attempt_count}\n"
                        f"elapsed={_fmt_duration(elapsed)}  eta={_fmt_duration(eta_sec)}\n"
                        f"avg completed speed={tags_per_min:,.2f} tags/min   avg/tag={_fmt_duration(avg_tag_sec)}\n"
                        f"avg attempt speed={attempts_per_min:,.2f} attempts/min   avg/attempt={_fmt_duration(avg_attempt_sec)}\n"
                        f"next retry in {_fmt_duration(sleep_for)}\n"
                        f"log={log_file}"
                    )

                time.sleep(sleep_for)
                continue

            batch = ready[:workers]
            batch_ids = {id(j) for j in batch}
            pending = [j for j in pending if id(j) not in batch_ids]

            with ThreadPoolExecutor(max_workers=workers) as pool:
                futures = [pool.submit(_worker, job) for job in batch]

                for fut in as_completed(futures):
                    res = fut.result()
                    job = res["job"]
                    attempt_count += 1
                    total_attempt_secs += res["secs"]

                    if res["ok"]:
                        one = res["df"]

                        # Always append, even if the tag had no usable rows,
                        # because fetch_one_point() now returns a full aligned empty column.
                        if ts_added:
                            one = one.drop(columns=["TS_LOCAL", "TS_UTC"], errors="ignore")
                        else:
                            ts_added = True

                        parts.append(one)
                        completed_ok += 1

                    else:
                        job["_attempts"] += 1

                        if job["_attempts"] >= max_attempts:
                            completed_fail += 1
                            _write_live_log(
                                "FAILED",
                                attempt=job["_attempts"],
                                name=str(job.get("Name", "")),
                                label=str(job.get("_label", "")),
                                path=str(job.get("Path", "")),
                                url="",
                                message=(
                                    f"unable to extract after {max_attempts} attempts; "
                                    f"last error: {res['error']}"
                                ),
                            )
                            errors.append({
                                "Name": job.get("Name"),
                                "Path": job.get("Path"),
                                "Descriptor": job.get("Descriptor"),
                                "EngineeringUnits": job.get("EngineeringUnits"),
                                "PointType": job.get("PointType"),
                                "Label": job.get("_label"),
                                "Attempts": job["_attempts"],
                                "Error": (
                                    f"Unable to extract after {max_attempts} attempts. "
                                    f"Last error: {res['error']}"
                                ),
                            })
                        else:
                            workers = max(min_workers, workers - 1)
                            job["_ready_at"] = time.time() + retry_wait_seconds
                            _write_live_log(
                                "RETRY",
                                attempt=job["_attempts"],
                                name=str(job.get("Name", "")),
                                label=str(job.get("_label", "")),
                                path=str(job.get("Path", "")),
                                url="",
                                message=(
                                    f"retry in {retry_wait_seconds}s; "
                                    f"workers reduced to {workers}; error: {res['error']}"
                                ),
                            )
                            pending.append(job)

                    if verbose and (
                        attempt_count - last_status_attempts >= status_every
                        or (completed_ok + completed_fail) == total
                    ):
                        elapsed = time.perf_counter() - t_start
                        done_total = completed_ok + completed_fail
                        tags_per_min = done_total / elapsed * 60.0 if done_total > 0 and elapsed > 0 else 0.0
                        avg_tag_sec = elapsed / done_total if done_total > 0 else None
                        eta_sec = len(pending) * avg_tag_sec if avg_tag_sec else None
                        attempts_per_min = attempt_count / elapsed * 60.0 if elapsed > 0 else 0.0
                        avg_attempt_sec = total_attempt_secs / attempt_count if attempt_count > 0 else None

                        _show_status(
                            f"[extracting]\n"
                            f"done={done_total}/{total}  ok={completed_ok}  failed={completed_fail}  pending={len(pending)}\n"
                            f"workers={workers}  attempts={attempt_count}\n"
                            f"elapsed={_fmt_duration(elapsed)}  eta={_fmt_duration(eta_sec)}\n"
                            f"avg completed speed={tags_per_min:,.2f} tags/min   avg/tag={_fmt_duration(avg_tag_sec)}\n"
                            f"avg attempt speed={attempts_per_min:,.2f} attempts/min   avg/attempt={_fmt_duration(avg_attempt_sec)}\n"
                            f"last attempt={_fmt_duration(res['secs'])}\n"
                            f"log={log_file}"
                        )
                        last_status_attempts = attempt_count

        # Merge everything
        if parts:
            wide_df = pd.concat(parts, axis=1, join="outer").sort_index()
            ts_cols = [c for c in ["TS_LOCAL", "TS_UTC"] if c in wide_df.columns]
            value_cols = [c for c in wide_df.columns if c not in ts_cols]
            wide_df = wide_df[ts_cols + sorted(value_cols)]
        else:
            wide_df = pd.DataFrame(index=master_index)
            wide_df.index.name = "UTC_Index"
            wide_df["TS_UTC"] = wide_df.index.strftime("%Y-%m-%d %H:%M:%S")
            wide_df["TS_LOCAL"] = wide_df.index.tz_localize(None).strftime("%Y-%m-%d %H:%M:%S")

        errors_df = pd.DataFrame(errors)

        if verbose:
            elapsed = time.perf_counter() - t_start
            done_total = completed_ok + completed_fail
            tags_per_min = done_total / elapsed * 60.0 if done_total > 0 and elapsed > 0 else 0.0
            avg_tag_sec = elapsed / done_total if done_total > 0 else None
            attempts_per_min = attempt_count / elapsed * 60.0 if elapsed > 0 else 0.0
            avg_attempt_sec = total_attempt_secs / attempt_count if attempt_count > 0 else None

            _show_status(
                f"[finished]\n"
                f"done={done_total}/{total}  ok={completed_ok}  failed={completed_fail}\n"
                f"workers={workers}  attempts={attempt_count}\n"
                f"elapsed={_fmt_duration(elapsed)}\n"
                f"avg completed speed={tags_per_min:,.2f} tags/min   avg/tag={_fmt_duration(avg_tag_sec)}\n"
                f"avg attempt speed={attempts_per_min:,.2f} attempts/min   avg/attempt={_fmt_duration(avg_attempt_sec)}\n"
                f"log={log_file}"
            )

        if output_parquet:
            save_to_parquet(wide_df, errors_df, output_parquet)

        return wide_df, errors_df

    finally:
        _close_live_log()
        _awake.stop()


In [ ]:
# ---------------------------------------------------------------------------
# 7) Example usage
# ---------------------------------------------------------------------------
server     = "https://piserver.companyweb.com/piwebapi/"
data_server = "PIDASERVER"


# A) Load or build tag index
all_points = load_index(
    data_server_name=data_server,
    server=server,
    reindex=False,
    max_results=200_000,
    verify=False,
    verbose=True,
)
print(f"Total points in index: {len(all_points)}")


[load_index] Loading cached index from 'BRUGPP01_index.xlsx' ...
[load_index] 11356 points loaded.
Total points in index: 11356


In [3]:
# B) Filter locally
points = filter_index(all_points, name_filter="*", description_filter="*")
print(f"Filtered points: {len(points)}")


[filter_index] name_filter '*': keep all 11356 (no filter applied).
[filter_index] description_filter '*': keep all 11356 (no filter applied).
Filtered points: 11356


In [ ]:
# C) Fetch and save to parquet
wide, errors = fetch_many_points(
    server=server,
    points_df=points[:1000],
    start="*-700d",
    end="*",
    interval="24h",
    sync_time="03:00:00",
    verify=False,
    verbose=True,
    output_parquet=data_server + "_data.parquet",
    log_file=data_server + "_live.log",
    max_workers=5,
    min_workers=1,
    max_attempts=5,
    retry_wait_seconds=10,
    status_every=5,
    paging_delay_ms=20,
)

print(f"\nShape : {wide.shape}")
print(f"Errors: {len(errors)}")


In [6]:
# D) Convert parquet -> Excel (run separately, after extraction is done)
parquet_to_excel(data_server + "_data.parquet")

[parquet_to_excel] Reading BRUGPP01_data.parquet ...
[parquet_to_excel] Shape: (701, 102)
[parquet_to_excel] Writing 701 rows in 1 sheet(s) -> BRUGPP01_data.xlsx ...
  Sheet 'Data': 701 rows
[parquet_to_excel] Done -> BRUGPP01_data.xlsx
